In [1]:
%load_ext autoreload
%autoreload 2

In [15]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import re
import scipy.optimize as opt
from tqdm import tqdm
import sympy as sp
from sympy.parsing.sympy_parser import parse_expr
from sympy import lambdify
from global_parameter import *
import torch
from torch import nn, optim

In [30]:
filename = 'pareto_low_adv_refit.csv'  
t_eq=pd.read_csv('/data/zj448/SR/Ultimate_paper/pareto_archive/'+filename)

df_full = pd.read_csv('SMBH_Data_03_06_24.csv',header=1)

paras=low_scatter_para_std

In [24]:
booleans=['ETG','Bar','Disk','Ring','Core','Multiple','Compactness','AGN','Pseudobulge','BCG','cD']
for b in booleans:
    df_full[b+'_std']=0
    
df=df_full[low_scatter_para_std].dropna(axis='index',how='any').copy()

In [25]:
t_eq

,complexity,loss,score,equation,sympy_format,lambda_format,number_constants,variables,number_variables,unique_number_variables,evolutions,iterations,fitting_format,num_fitting_variables,initial_constant_guess,LLL,intrinsic_scatter,refit_equation,refit_wrmse
0,1,4.906695,0.000000,x48,x48,PySRFunction(X=>x48),0,{'x48'},1,1,0,0,x48,0,[],-2684.924248,2.502863,x48,4.906695
1,15,0.088578,0.044147,((x12 - (exp(log10(exp(0.04766298070500929) / ...,x12 - (1.048817124146649/(x19 + 0.047662980705...,PySRFunction(X=>x12 - (1.048817124146649/(x19 ...,3,"{'x19', 'x29', 'x12'}",3,3,0,0,x12 - log10(x29) - exp(log10(p[1]/(x19 + p[0])...,3,"['0.04766298070500929', '1.048817124146649', '...",NaN,NaN,x12 - log10(x29) - exp(log10(-0.12211717766330...,NaN
2,16,0.088371,0.002337,((x12 - (exp(log10(exp(exp(-1.0423515630962228...,x12 - (exp(0.35262448748241617/x12)/x19)**(1/l...,PySRFunction(X=>x12 - (exp(0.35262448748241617...,2,"{'x19', 'x29', 'x12'}",4,3,0,0,x12 - log10(x29) - exp(log10(exp(p[0]/x12)/x19...,2,"['0.35262448748241617', '1.29244433001088']",-76.179376,0.424280,x12 - log10(x29) - exp(log10(exp(-892.51893396...,0.107443
3,17,0.086897,0.016823,((x12 - (exp(log10(exp(x29 / x12) / (x19 + 0.0...,x12 - (exp(x29/x12)/(x19 + 0.04766298070500929...,PySRFunction(X=>x12 - (exp(x29/x12)/(x19 + 0.0...,2,"{'x29', 'x12', 'x19'}",5,3,0,0,x12 - log10(x29) - exp(log10(exp(x29/x12)/(x19...,2,"['0.04766298070500929', '1.29244433001088']",-70.893942,0.421744,x12 - log10(x29) - exp(log10(exp(x29/x12)/(x19...,0.101308
4,19,0.086145,0.004346,((x12 - (exp(log10(exp(x29 / x12) / ((x19 - x1...,x12 - (exp(x29/x12)/(-x14 + x19 + 0.0476629807...,PySRFunction(X=>x12 - (exp(x29/x12)/(-x14 + x1...,2,"{'x29', 'x14', 'x12', 'x19'}",6,4,0,0,x12 - log10(x29) - exp(log10(exp(x29/x12)/(-x1...,2,"['0.04766298070500929', '1.29244433001088']",-74.219419,0.422749,x12 - log10(x29) - exp(log10(exp(x29/x12)/(-x1...,0.104474
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
786,13,0.015437,0.052914,((((x15 * 3.0860786087580303) - -2.72029684243...,3.0860786087580303*x15 + 2.835860812650755 - 2...,PySRFunction(X=>3.0860786087580303*x15 + 2.835...,3,"{'x26', 'x15', 'x29'}",4,3,4,2969,p[0]*x15 + p[1] - p[2]*x29/x26,3,"['3.08607860875803', '2.83586081265075', '2.0']",-6.342393,0.129297,2.214948689657893*x15 + 5.318061088249941 - 2....,0.012861
787,11,0.013242,0.190891,((x15 * (3.4412858435521843 - (x29 / (x26 + 0....,x15*(-x29/(x26 + 0.012180693250749317) + 3.441...,PySRFunction(X=>x15*(-x29/(x26 + 0.01218069325...,3,"{'x26', 'x15', 'x29'}",3,3,4,3493,x15*(-x29/(x26 + p[0]) + p[1]) + p[2],3,"['0.0121806932507493', '3.44128584355218', '2....",-2.118180,0.142504,x15*(-x29/(x26 + -0.30270296186787016) + 3.208...,0.015038
788,25,0.012863,0.002140,((((x15 * (3.4412858435521843 - (x29 / x26))) ...,x15*(3.4412858435521843 - x29/x26) + log(exp(1...,PySRFunction(X=>x15*(3.4412858435521843 - x29/...,6,"{'x0', 'x26', 'x15', 'x29'}",5,4,4,3494,x15*(p[2] - x29/x26) + log10(exp(p[1]*x0*exp(-...,5,"['0.8673260387054437', '1.613264904706933', '3...",-1.763859,0.127957,x15*(3.5326679489905857 - x29/x26) + log10(exp...,0.014635
789,22,0.011836,0.014948,(((x15 * ((3.5011432103496922 - (log10(((x46 -...,x15*(3.5011432103496922 - log(1.63819164201210...,PySRFunction(X=>x15*(3.5011432103496922 - log(...,6,"{'x26', 'x15', 'x46', 'x54', 'x29'}",5,5,4,3559,x15*(p[3] - log10(p[1]/(p[0]*x46 + p[5])**p[2]...,6,"['0.39262958179688978', '1.6381916420121015', ...",-2.134484,0.138010,x15*(3.6611160870021457 - log10(23186.27872631...,0.014096


In [6]:
def str2equ(equation):
    return lambdify(list(dict.fromkeys(re.findall(r'\bx\d+',equation))),equation)

In [41]:
t_eq.iloc[1]['refit_equation']

'x12 - log10(x29) - exp(log10(-0.12211717766330336/(x19 + 0.1550515639475048))) - 1.2490450126043844'

In [14]:
row[1]

complexity                                                                20
loss                                                                0.010133
score                                                               0.060767
equation                   ((((x15 / 0.31852133710431946) + (x23 ^ -0.000...
sympy_format               3.1395071020704912*x15 + x23**(-0.000395449818...
lambda_format              PySRFunction(X=>3.1395071020704912*x15 + x23**...
number_constants                                                           4
variables                                {'x42', 'x19', 'x23', 'x39', 'x15'}
number_variables                                                           6
unique_number_variables                                                    5
evolutions                                                                 4
iterations                                                              4632
fitting_format             p[3]*x15 + x23**(-p[0]) + (x39 - x42)/(exp(x19...

In [44]:
number_matching_pattern = r"(?<![a-zA-Z0-9_.])[+-]?(\d+\.\d+|\.\d+|\d+\.|\d+)(?:[eE][-+]?\d+)?"

equation = row[1]['refit_equation']

# Find all unique matches first to avoid duplicates and incorrect indexing
constants = list(set(re.findall(number_matching_pattern, equation)))
# Sort constants by their length in descending order to replace longer numbers first, preventing partial replacement issues
constants.sort(key=len, reverse=True)

def replace_with_variable(match):
    # Find the matched number in the constants list and get its index
    number = match.group(1)
    index = constants.index(number)

    # Check if the match includes a minus sign
    if match.group(0).startswith('-'):
        sign = '-'
    else:
        sign = ''
    return f'{sign}p[{index}]'

equation = re.sub(number_matching_pattern, replace_with_variable, equation)


In [46]:
row[1]['refit_equation']

'3.0440584383966294*x15 + x23**(-0.001880592940607885) + (x39 - x42)/(exp(x19) - 0.20561326182727518/x23) - 0.11337624799367435'

In [47]:
constants

['0.001880592940607885',
 '0.11337624799367435',
 '0.20561326182727518',
 '3.0440584383966294']

In [45]:
equation

'p[3]*x15 + x23**(-p[0]) + (x39 - x42)/(exp(x19) - p[2]/x23) - p[1]'

In [82]:
eq = equation
for x_index in x_indexs:
    eq = eq.replace(f'x{x_index}', f'self.x{x_index}[i]')

eq

'self.x15[i]*(-self.x29[i]/(self.x26[i] + self.p0) + self.p1)'

In [ ]:
# Convert the dataframe to PyTorch tensors
M_BH = torch.tensor(df['M_BH'].values, dtype=torch.float32)
M_BH_std_sym = torch.tensor(df['M_BH_std_sym'].values, dtype=torch.float32)

logR10 = torch.tensor(df['logR10'].values, dtype=torch.float32)
logR10_std = torch.tensor(df['logR10_std'].values, dtype=torch.float32)
log_sigma0 = torch.tensor(df['log_sigma0'].values, dtype=torch.float32)
log_sigma0_std = torch.tensor(df['log_sigma0_std'].values, dtype=torch.float32)

# Define the model
class Model(nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.p0 = nn.Parameter(torch.tensor(1.173655460551097, dtype=torch.float32))
        self.p1 = nn.Parameter(torch.tensor(1.9838167739222974, dtype=torch.float32))
        self.p2 = nn.Parameter(torch.tensor(1.5299898769767926, dtype=torch.float32))
        self.p3 = nn.Parameter(torch.tensor(t_eq.iloc[65]['intrinsic_scatter'], dtype=torch.float32))
        self.p4 = nn.Parameter(logR10.clone())
        self.p5 = nn.Parameter(log_sigma0.clone())

    def forward(self, i):
        return self.p0 ** self.p4[i] + self.p1 * self.p5[i] ** self.p2

# Instantiate the model
model = Model()

# Define the loss function
def loglikelihood():
    term0 = torch.log(torch.tensor(2 * torch.pi)) * len(df) * (t_eq.iloc[65]['unique_number_variables'] + 1)

    term1 = (torch.log(M_BH_std_sym**2 + model.p3**2)).sum()
    term1 += torch.log(logR10_std**2).sum()
    term1 += torch.log(log_sigma0_std**2).sum()
    
    term2 = ((M_BH - model(torch.arange(len(df))))**2 / (M_BH_std_sym**2 + model.p3**2)).sum()

    term3 = (((log_sigma0 - model.p5) / log_sigma0_std)**2).sum()
    term3 += (((logR10 - model.p4) / logR10_std)**2).sum()

    return term0 + term1 + term2 + term3

# Define the optimizer
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training loop
for epoch in range(10000):
    optimizer.zero_grad()
    loss = loglikelihood()
    loss.backward()
    optimizer.step()
    if epoch % 100 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item()}', model.p0.item(), model.p1.item(), model.p2.item(), model.p3.item())

# Get the optimized parameters
optimized_params = {name: param.data for name, param in model.named_parameters()}
print(optimized_params)

/home/zj448/miniconda3/lib/python3.9/site-packages/torch/autograd/__init__.py:251: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [93]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /croot/pytorch-select_1700158693612/work/aten/src/ATen/native/Resize.cpp:28.)
  Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 0, Loss: -872.4113159179688 1.1636558771133423 1.9738171100616455 1.5399895906448364 0.44972172379493713
Epoch 100, Loss: -878.4766845703125 1.1733146905899048 1.9517251253128052 1.5510613918304443 0.4097220003604889
Epoch 200, Loss: -878.4879760742188 1.1721328496932983 1.9401181936264038 1.5583935976028442 0.4096180200576782
Epoch 300, Loss: -878.4881591796875 1.1717450618743896 1.9381600618362427 1.559680461883545 0.40959277749061584
Epoch 400, Loss: -878.4881591796875 1.1717193126678467 1.9380286931991577 1.5597667694091797 0.4095909893512726
Epoch 500, Loss: -878.4881591796875 1.171718716621399 1.9380260705947876 1.5597689151763916 0.4095909595489502
Epoch 600, Loss: -878.4881591796875 1.171718716621399 1.9380258321762085 1.559768795967102 0.4095909297466278
Epoch 700, Loss: -878.4881591796875 1.171718955039978 1.938025951385498 1.559768795967102 0.4095909893512726
Epoch 800, Loss: -878.4881591796875 1.171718716621399 1.9380260705947876 1.559768795967102 0.40959107875823975


In [172]:
number_matching_pattern = r"(?<![a-zA-Z0-9_.])[+-]?(\d+\.\d+|\.\d+|\d+\.|\d+)(?:[eE][-+]?\d+)?"

from torch import log10
from torch import exp

M_BH = torch.tensor(df['M_BH'].values, dtype=torch.float64)
M_BH_std_sym = torch.tensor(df['M_BH_std_sym'].values, dtype=torch.float64)

for row in tqdm(t_eq[319:320].iterrows(),total=len(t_eq)):
    
    equation = row[1]['refit_equation']

    # Find all unique matches first to avoid duplicates and incorrect indexing
    constants = list(set(re.findall(number_matching_pattern, equation)))
    # Sort constants by their length in descending order to replace longer numbers first, preventing partial replacement issues
    constants.sort(key=len, reverse=True)

    def replace_with_variable(match):
        # Find the matched number in the constants list and get its index
        number = match.group(1)
        index = constants.index(number)

        # Check if the match includes a minus sign
        if match.group(0).startswith('-'):
            sign = '-'
        else:
            sign = ''
        return f'{sign}self.p{index}'

    equation = re.sub(number_matching_pattern, replace_with_variable, equation)

    # get x_indexs
    x_indexs = re.findall(r'x(\d+)', equation)
    x_indexs = list(dict.fromkeys(x_indexs))
    x_indexs = [int(x_index) for x_index in x_indexs]

    # construct tensors for initial position of x
    x=[]
    x_stds=[]
    for x_index in x_indexs:
        x.append(torch.tensor(df[paras[x_index]].values, dtype=torch.float64))
        x_stds.append(torch.tensor(df[paras[x_index]+'_std'].values, dtype=torch.float64))

    # Define the model
    class Model(nn.Module):
        
        def __init__(self):
            # assign initial values to parameters self.p[i], self.x[i] and self.intrinsic_scatter
            super(Model, self).__init__()
            for i in range(len(constants)):
                setattr(self, f'p{i}', nn.Parameter(torch.tensor(float(constants[i]), dtype=torch.float64)))
            for i,x_index in enumerate(x_indexs):
                if x_stds[i].sum() != 0:
                    setattr(self, f'x{x_index}', nn.Parameter(x[i].clone()))
                else:
                    setattr(self, f'x{x_index}', x[i].clone())
            self.intrinsic_scatter = nn.Parameter(torch.tensor(row[1]['intrinsic_scatter'], dtype=torch.float64))
        
        def forward(self,i):
            # calculate the model prediction
            # replace x[i] with self.x[i]
            eq = equation
            for x_index in x_indexs:
                eq = eq.replace(f'x{x_index}', f'self.x{x_index}[i]')
            # replace exp with torch.exp
            
            # evaluate the equation
            return eval(eq)
        
    model = Model()

    # Define the loss function
    def loglikelihood():
        # term0
        term0 = torch.log(torch.tensor(2 * torch.pi)) * len(df) * (row[1]['unique_number_variables'] + 1)

        #term1
        term1 = (torch.log(M_BH_std_sym**2 + model.intrinsic_scatter**2)).sum()
        for j in range(len(x_indexs)):
            if x_stds[j].sum() != 0:
                term1 += (torch.log(x_stds[j]**2)).sum()

        #term2
        term2 = ((M_BH - model(torch.arange(len(df))))**2 / (M_BH_std_sym**2 + model.intrinsic_scatter**2)).sum()
        
        #term3
        term3 = 0
        for j in range(len(x_indexs)):
            if x_stds[j].sum() != 0:
                term3 += (((x[j] - getattr(model, f'x{x_indexs[j]}')) / x_stds[j])**2).sum()

        return term0 + term1 + term2 + term3
        
    # Define the optimizer
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    # Training loop
    for epoch in range(10000):
        optimizer.zero_grad()
        loss = loglikelihood()
        loss.backward()
        optimizer.step()
        if epoch % 100 == 0:
            print(f'Epoch {epoch}, Loss: {loss.item()}', [getattr(model, f'p{i}').item() for i in range(len(constants))], model.intrinsic_scatter.item())

    # Get the optimized parameters
    optimized_params = {name: param.data for name, param in model.named_parameters()}
    print(optimized_params)

  0%|          | 0/791 [00:00<?, ?it/s]

Epoch 0, Loss: -530.4456725753321 [2.471107196607295, 4.678610513853765] 0.43993101267236134
Epoch 100, Loss: -536.5543358481154 [2.5780084007431627, 4.7426293244069155] 0.4194272908077283
Epoch 200, Loss: -536.621186973882 [2.6838173208077527, 4.789094088264597] 0.41890068079939746
Epoch 300, Loss: -536.6295645272097 [2.726790299235739, 4.807945464943311] 0.4186825313316814
Epoch 400, Loss: -536.6300210815922 [2.7375904300348055, 4.812683119209208] 0.41863202046891235
Epoch 500, Loss: -536.6300321306788 [2.7393550684518138, 4.813457200837655] 0.4186239249917025
Epoch 600, Loss: -536.6300322464225 [2.7395417993940385, 4.813539112703054] 0.4186230709366491
Epoch 700, Loss: -536.6300322469051 [2.7395541255260065, 4.813544519714517] 0.4186230145778098
Epoch 800, Loss: -536.6300322469057 [2.73955459424211, 4.813544725322677] 0.41862301243473915
Epoch 900, Loss: -536.6300322469057 [2.7395546029045508, 4.8135447291225635] 0.41862301239513255
Epoch 1000, Loss: -536.6300322469057 [2.7395546029

  0%|          | 1/791 [00:08<1:51:54,  8.50s/it]

Epoch 9800, Loss: -536.6060159009476 [2.7395138394107583, 4.813503881050615] 0.41855267708185917
Epoch 9900, Loss: -536.4531341763259 [2.739427546010947, 4.8135582492973805] 0.4191972429788425
{'p0': tensor(2.73942664, dtype=torch.float64), 'p1': tensor(4.81357213, dtype=torch.float64), 'x15': tensor([2.47204969, 2.51492180, 1.57448954, 2.37418672, 2.29571323, 2.29285681,
        2.17933530, 2.39001445, 2.34799520, 2.46937967, 2.25454999, 2.52071785,
        2.42530253, 2.52211476, 2.14921575, 2.18685632, 2.28181132, 2.49386451,
        2.41536537, 2.31545802, 2.13468284, 2.30648950, 2.15813770, 2.37401839,
        2.01753288, 2.33090518, 2.34639806, 2.28923794, 2.33465778, 2.49008087,
        2.39124260, 2.41980224, 2.23990106, 2.47238099, 2.46595819, 2.04708015,
        2.38119061, 2.25850606, 2.10879806, 2.44307373, 2.23990160, 2.06721791,
        2.23449105, 2.45012425, 2.25182087, 2.50959665, 2.35219098, 2.39853215,
        2.19376053, 2.04991128, 2.14791715, 2.35774023, 2.5192752

In [ ]:
# equation, loss, array of fitted constants, intrinsic scatter, array of fitted x positions, 2nd derivative of loss 